# AUC ROC scores

This notebook evaluates the performance of a trained linear classifier using embeddings from various pretrained audio models (**BirdNET**, **YAMNet**, **Perch**, **HawkEars**, **BirdSetConvNeXT**, **BirdSetEfficientNetB1**, **RanaSierraeCNN**). 

It performs the following steps:
* Loads precomputed embeddings and corresponding labels
* Splits data into training and validation sets
* Loads the best model checkpoint for the specified embedding model
* Evaluates the model using accuracy, F1 score, precision, recall, and AUC-ROC (macro, weighted, and per-class)
* Visualizes the confusion matrix and ROC curves (for small class counts)
* Saves all evaluation results to a structured JSON file

Update the `model_name` variable at the top of the notebook to rerun this evaluation for a different model.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
import glob
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from birdclef.torch.workflow import (
    load_preprocess_data,
    perform_train_test_split,
    label_index_mapping,
)
from birdclef.torch.data import BirdDataModule
from birdclef.torch.model import LinearClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_curve,
    auc,
)

Seed set to 42


In [3]:
model_name = "BirdNET"
dataset_name = "train_audio-infer-soundscape"
project_dir = "/storage/coda1/p-dsgt_clef2025/0/shared/birdclef"
input_path = f"{project_dir}/data/2025/{dataset_name}/{model_name}/parts/embed/"
output_path = f"{project_dir}/models/2025/v2/{model_name}"

# training parameters (should match the values used during training)
learning_rate = 1e-3
batch_size = 64

In [4]:
# load and preprocess data
df = load_preprocess_data(input_path, model_name)

# train/test split
X_train, X_test, y_train, y_test = perform_train_test_split(df)

# create label index mapping
label_to_idx = label_index_mapping(df, output_path)
num_classes = len(label_to_idx)

# instantiate DataModule
data_module = BirdDataModule(
    X_train, X_test, y_train, y_test, label_to_idx, batch_size=batch_size
)

# instantiate model
input_dim = X_train.shape[1]
model = LinearClassifier(input_dim=input_dim, num_classes=num_classes, lr=learning_rate)

DataFrame shape: (939116, 2)
Embedding size: 1024
X_train, X_test shape: ((751292, 1024), (187824, 1024))
y_train, y_test shape: ((751292,), (187824,))


In [5]:
def find_best_checkpoint(checkpoint_dir):
    """Find the best checkpoint file in the checkpoint directory"""
    checkpoint_pattern = os.path.join(
        checkpoint_dir, f"best-checkpoint-{model_name}-*.ckpt"
    )
    checkpoint_files = glob.glob(checkpoint_pattern)

    if not checkpoint_files:
        raise FileNotFoundError(f"No checkpoint files found in {checkpoint_dir}")

    # Return the most recent checkpoint (or you could implement logic to find the one with best validation loss)
    best_checkpoint = max(checkpoint_files, key=os.path.getmtime)
    print(f"Loading checkpoint: {best_checkpoint}")
    return best_checkpoint

In [6]:
# Load the best checkpoint
checkpoint_dir = f"{output_path}/checkpoints"
best_checkpoint_path = find_best_checkpoint(checkpoint_dir)

# Load the trained model
trained_model = LinearClassifier.load_from_checkpoint(
    best_checkpoint_path, input_dim=input_dim, num_classes=num_classes, lr=learning_rate
)

# Set model to evaluation mode
trained_model.eval()
print(f"Model loaded successfully from {best_checkpoint_path}")

Loading checkpoint: /storage/coda1/p-dsgt_clef2025/0/shared/birdclef/models/2025/v2/BirdNET/checkpoints/best-checkpoint-BirdNET-epoch=04-val_loss=0.61.ckpt
Model loaded successfully from /storage/coda1/p-dsgt_clef2025/0/shared/birdclef/models/2025/v2/BirdNET/checkpoints/best-checkpoint-BirdNET-epoch=04-val_loss=0.61.ckpt


In [7]:
def get_predictions(model, X, y, label_to_idx, batch_size=64):
    """Get predictions and probabilities from the model"""
    from torch.utils.data import DataLoader
    from birdclef.torch.data import BirdDataset

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Create dataset and dataloader
    dataset = BirdDataset(X, y, label_to_idx)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_preds = []
    all_probs = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            # Move tensors to the same device as the model
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            # Get model output
            logits = model(batch_x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    return np.array(all_preds), np.array(all_probs), np.array(all_labels)

In [9]:
# Evaluate on training set
print("=== TRAINING SET EVALUATION ===")
train_preds, train_probs, train_labels = get_predictions(
    trained_model, X_train, y_train, label_to_idx, batch_size
)

# Basic metrics
train_accuracy = accuracy_score(train_labels, train_preds)
train_f1_macro = f1_score(train_labels, train_preds, average="macro")
train_f1_micro = f1_score(train_labels, train_preds, average="micro")
train_f1_weighted = f1_score(train_labels, train_preds, average="weighted")
train_recall_macro = recall_score(train_labels, train_preds, average="macro")
train_precision_macro = precision_score(train_labels, train_preds, average="macro")

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Training F1 (Macro): {train_f1_macro:.4f}")
print(f"Training F1 (Micro): {train_f1_micro:.4f}")
print(f"Training F1 (Weighted): {train_f1_weighted:.4f}")
print(f"Training Recall (Macro): {train_recall_macro:.4f}")
print(f"Training Precision (Macro): {train_precision_macro:.4f}")

# AUC-ROC scores
train_auc_macro = roc_auc_score(
    train_labels, train_probs, multi_class="ovr", average="macro"
)
train_auc_weighted = roc_auc_score(
    train_labels, train_probs, multi_class="ovr", average="weighted"
)
print(f"Training AUC-ROC (Macro): {train_auc_macro:.4f}")
print(f"Training AUC-ROC (Weighted): {train_auc_weighted:.4f}")

=== TRAINING SET EVALUATION ===
Training Accuracy: 0.9076
Training F1 (Macro): 0.9128
Training F1 (Micro): 0.9076
Training F1 (Weighted): 0.9078
Training Recall (Macro): 0.9184
Training Precision (Macro): 0.9138
Training AUC-ROC (Macro): 0.9995
Training AUC-ROC (Weighted): 0.9988


In [10]:
# Evaluate on validation/test set
print("\n=== VALIDATION/TEST SET EVALUATION ===")
val_preds, val_probs, val_labels = get_predictions(
    trained_model, X_test, y_test, label_to_idx, batch_size
)

# Basic metrics
val_accuracy = accuracy_score(val_labels, val_preds)
val_f1_macro = f1_score(val_labels, val_preds, average="macro")
val_f1_micro = f1_score(val_labels, val_preds, average="micro")
val_f1_weighted = f1_score(val_labels, val_preds, average="weighted")
val_recall_macro = recall_score(val_labels, val_preds, average="macro")
val_precision_macro = precision_score(val_labels, val_preds, average="macro")

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation F1 (Macro): {val_f1_macro:.4f}")
print(f"Validation F1 (Micro): {val_f1_micro:.4f}")
print(f"Validation F1 (Weighted): {val_f1_weighted:.4f}")
print(f"Validation Recall (Macro): {val_recall_macro:.4f}")
print(f"Validation Precision (Macro): {val_precision_macro:.4f}")

# AUC-ROC scores
val_auc_macro = roc_auc_score(val_labels, val_probs, multi_class="ovr", average="macro")
val_auc_weighted = roc_auc_score(
    val_labels, val_probs, multi_class="ovr", average="weighted"
)
print(f"Validation AUC-ROC (Macro): {val_auc_macro:.4f}")
print(f"Validation AUC-ROC (Weighted): {val_auc_weighted:.4f}")


=== VALIDATION/TEST SET EVALUATION ===
Validation Accuracy: 0.8546
Validation F1 (Macro): 0.8572
Validation F1 (Micro): 0.8546
Validation F1 (Weighted): 0.8548
Validation Recall (Macro): 0.8577
Validation Precision (Macro): 0.8657
Validation AUC-ROC (Macro): 0.9982
Validation AUC-ROC (Weighted): 0.9965


In [11]:
# Detailed classification report
print("\n=== DETAILED CLASSIFICATION REPORT (VALIDATION SET) ===")

# Create reverse mapping from index to label
idx_to_label = {v: k for k, v in label_to_idx.items()}
class_names = [idx_to_label[i] for i in range(len(label_to_idx))]

# Print classification report
report = classification_report(
    val_labels, val_preds, target_names=class_names, digits=4
)
print(report)


=== DETAILED CLASSIFICATION REPORT (VALIDATION SET) ===
              precision    recall  f1-score   support

     1139490     0.3191    0.3947    0.3529        38
     1192948     0.2886    0.7073    0.4099        82
     1194042     0.7143    0.8824    0.7895        17
      126247     1.0000    1.0000    1.0000        17
     1346504     0.9825    0.9032    0.9412        62
      134933     0.9286    0.9630    0.9455        27
      135045     0.9801    0.9801    0.9801       251
     1462711     0.6190    0.2167    0.3210        60
     1462737     0.5484    0.3643    0.4378       140
       21038     1.0000    1.0000    1.0000        78
       21116     1.0000    1.0000    1.0000         1
       21211     0.9097    0.9403    0.9248       268
       22333     0.9931    0.9862    0.9896       145
       22973     0.9354    0.8941    0.9143       340
       22976     0.8603    0.8701    0.8652       177
       24272     0.9688    0.8857    0.9254        35
       24292     0.9529 

In [13]:
# Per-class AUC-ROC scores
print("\n=== PER-CLASS AUC-ROC SCORES ===")

# Convert labels to one-hot encoding for per-class AUC calculation
from sklearn.preprocessing import label_binarize

val_labels_binarized = label_binarize(val_labels, classes=range(num_classes))

per_class_auc = {}
fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_classes):
    if (
        np.sum(val_labels_binarized[:, i]) > 0
    ):  # Check if class exists in validation set
        fpr[i], tpr[i], _ = roc_curve(val_labels_binarized[:, i], val_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        per_class_auc[class_names[i]] = roc_auc[i]

        # Print only if not too many classes
        if len(class_names) <= 20:
            print(f"{class_names[i]}: {roc_auc[i]:.4f}")

# Summary statistics
auc_values = list(per_class_auc.values())
print(f"\nAUC-ROC Summary:")
print(f"Mean AUC: {np.mean(auc_values):.4f}")
print(f"Std AUC: {np.std(auc_values):.4f}")
print(f"Min AUC: {np.min(auc_values):.4f}")
print(f"Max AUC: {np.max(auc_values):.4f}")
print(f"Number of classes with AUC > 0.8: {sum(1 for auc in auc_values if auc > 0.8)}")
print(f"Number of classes with AUC > 0.9: {sum(1 for auc in auc_values if auc > 0.9)}")


=== PER-CLASS AUC-ROC SCORES ===

AUC-ROC Summary:
Mean AUC: 0.9982
Std AUC: 0.0021
Min AUC: 0.9894
Max AUC: 1.0000
Number of classes with AUC > 0.8: 205
Number of classes with AUC > 0.9: 205


In [15]:
# Save evaluation results
evaluation_results = {
    "model_name": model_name,
    "checkpoint_path": best_checkpoint_path,
    "num_classes": num_classes,
    "training_metrics": {
        "accuracy": float(train_accuracy),
        "f1_macro": float(train_f1_macro),
        "f1_micro": float(train_f1_micro),
        "f1_weighted": float(train_f1_weighted),
        "recall_macro": float(train_recall_macro),
        "precision_macro": float(train_precision_macro),
    },
    "validation_metrics": {
        "accuracy": float(val_accuracy),
        "f1_macro": float(val_f1_macro),
        "f1_micro": float(val_f1_micro),
        "f1_weighted": float(val_f1_weighted),
        "recall_macro": float(val_recall_macro),
        "precision_macro": float(val_precision_macro),
    },
    "per_class_auc": per_class_auc,
    "auc_summary": {
        "mean_auc": float(np.mean(auc_values)),
        "std_auc": float(np.std(auc_values)),
        "min_auc": float(np.min(auc_values)),
        "max_auc": float(np.max(auc_values)),
        "classes_auc_gt_08": int(sum(1 for auc in auc_values if auc > 0.8)),
        "classes_auc_gt_09": int(sum(1 for auc in auc_values if auc > 0.9)),
    },
}

# Add overall AUC scores
# Train
evaluation_results["training_metrics"]["auc_macro"] = float(train_auc_macro)
evaluation_results["training_metrics"]["auc_weighted"] = float(train_auc_weighted)

# Validation
evaluation_results["validation_metrics"]["auc_macro"] = float(val_auc_macro)
evaluation_results["validation_metrics"]["auc_weighted"] = float(val_auc_weighted)

# Save results to file
results_path = f"{output_path}/evaluation_results.json"
os.makedirs(os.path.dirname(results_path), exist_ok=True)
with open(results_path, "w") as f:
    json.dump(evaluation_results, f, indent=2)

print(f"\nEvaluation results saved to: {results_path}")

# Print final summary
print("\n" + "=" * 50)
print("FINAL EVALUATION SUMMARY")
print("=" * 50)
print(f"Model: {model_name}")
print(f"Number of classes: {num_classes}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation F1 (Macro): {val_f1_macro:.4f}")
if "val_auc_macro" in locals():
    print(f"Validation AUC-ROC (Macro): {val_auc_macro:.4f}")
print(f"Mean per-class AUC: {np.mean(auc_values):.4f}")
print(
    f"Classes with AUC > 0.8: {sum(1 for auc in auc_values if auc > 0.8)}/{len(auc_values)}"
)
print("=" * 50)


Evaluation results saved to: /storage/coda1/p-dsgt_clef2025/0/shared/birdclef/models/2025/v2/BirdNET/evaluation_results.json

FINAL EVALUATION SUMMARY
Model: BirdNET
Number of classes: 205
Validation Accuracy: 0.8546
Validation F1 (Macro): 0.8572
Validation AUC-ROC (Macro): 0.9982
Mean per-class AUC: 0.9982
Classes with AUC > 0.8: 205/205
